# Speexx — run BOTH (bot + panel) in one go

Colab / Kaggle companion to the repo's `run_all.sh`. Starts the panel (with a
public URL) and the AutoCycle bot for every account, then prints both URLs.

Secrets (Kaggle Secrets / Colab userdata): `SPEEXX_ACCOUNTS` (one
`email password` per line), `NGROK_AUTHTOKEN` (for the panel's public URL).
Never store or print secrets in the notebook.

In [ ]:
# 1) Locate the project (Kaggle datasets are read-only — copy to /kaggle/working)
import os, glob, shutil

proj = os.getcwd()
if not os.path.exists("run_all.sh"):
    hits = glob.glob("/kaggle/input/**/run_all.sh", recursive=True)
    if hits:
        src = os.path.dirname(hits[0])
        dst = "/kaggle/working/" + os.path.basename(src)
        if not os.path.exists(dst):
            shutil.copytree(src, dst)
        proj = dst
    else:
        raise SystemExit("run_all.sh not found — upload the repo and cd into it")
os.chdir(proj)
print("project dir:", proj)


In [ ]:
# 2) Secrets → environment + Accounts.md (never printed)
import os, pathlib

def _secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return None

accounts = _secret("SPEEXX_ACCOUNTS") or ""
token = _secret("NGROK_AUTHTOKEN") or ""
admin_pw = _secret("SPEEXX_PANEL_ADMIN_PASSWORD") or ""

if accounts.strip():
    pathlib.Path("Speexx-sonsole-cloud/Accounts.md").write_text(accounts.strip() + "\n")
    print("Accounts.md written")
else:
    print("SPEEXX_ACCOUNTS secret not set — run_all.sh will refuse to start")
if token:
    os.environ["NGROK_AUTHTOKEN"] = token
    print("NGROK_AUTHTOKEN: set")
else:
    print("NGROK_AUTHTOKEN: not set — panel runs without a public URL")
if admin_pw:
    os.environ["SPEEXX_PANEL_ADMIN_PASSWORD"] = admin_pw
    os.environ.setdefault("SPEEXX_PANEL_ADMIN_USERNAME", "admin")
    print("first admin will be bootstrapped")


In [ ]:
# 3) Launch both. 12600 = 3.5 h bot window.
import subprocess, sys
r = subprocess.run(["bash", "run_all.sh", "12600"])
if r.returncode != 0:
    sys.exit("run_all.sh failed — see the output above")

In [ ]:
# 4) Monitor — the bot console log is what shows what happened in the page
!tail -n 40 ~/speexx_autocycle_multi/session1_*.log.console 2>/dev/null || echo "no session log yet"

In [ ]:
# 5) Stop both when done
!bash run_all.sh stop